# Chromatogram Analysis: Sanger Sequencing in Practice

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain how Sanger (dideoxy chain termination) sequencing works
- Read `.ab1` chromatogram files with BioPython
- Plot and interpret trace data with matplotlib
- Work with Phred quality scores and base calling
- Identify mixed bases, heterozygous positions, and low-quality regions
- Understand assembly of multiple Sanger reads
- Know when to use Sanger sequencing vs NGS in a modern lab

---

[← Previous: Nucleic Acid Structure](../../Tier_2_Core_Bioinformatics/08_Nucleic_Acid_Structure/01_nucleic_acid_structure.ipynb) | [Next: Sequence Motifs and Protein Domains →](../../Tier_2_Core_Bioinformatics/10_Sequence_Motifs_and_Domains/01_sequence_motifs_and_domains.ipynb)

---

---

## 1. How Sanger Sequencing Works

### The Dideoxy Chain Termination Method

Frederick Sanger's method (1977, Nobel Prize 1980) exploits a subtle chemical difference
between normal deoxynucleotides (dNTPs) and **dideoxynucleotides (ddNTPs)**.

```
Normal dNTP (e.g. dATP):           Dideoxy ddNTP (e.g. ddATP):
       O                                   O
       ||                                  ||
   O-P-O-5'CH2  Base                   O-P-O-5'CH2  Base
       ||     \   |                       ||     \   |
       O       O--+                        O       O--+
              /                                   /
          3'-OH  <-- extension continues      3'-H  <-- STOP! No 3'-OH
```

**Key idea:** A ddNTP lacks the 3'-OH group required for the next phosphodiester bond.
When DNA polymerase incorporates a ddNTP, the growing chain **terminates** at that position.

### Modern Sanger Sequencing Workflow

1. **Reaction mix**: template DNA + primer + DNA polymerase + dNTPs (high concentration) + four fluorescently-labeled ddNTPs (low concentration, each color = one base)
2. **PCR cycling**: produces a population of fragments of every possible length, each terminated by a colored ddNTP
3. **Capillary electrophoresis**: fragments migrate through a polymer-filled capillary; shorter fragments travel faster
4. **Laser detection**: as fragments pass a detector window, laser excites the fluorescent label and the emission color reveals the terminal base
5. **Electropherogram**: software plots fluorescence intensity vs. time, producing the familiar four-color trace

```
Template:  3'-TAGCAGTCGATC-5'
Primer:    5'-ATC...

Some terminated fragments:
  5'-ATCG*           (4 bp, ddG - black)
  5'-ATCGT*          (5 bp, ddT - red)
  5'-ATCGTC*         (6 bp, ddC - blue)
  5'-ATCGTCA*        (7 bp, ddA - green)
  ...

Capillary electrophoresis separates by size:
Size -->  4    5    6    7    8    9   10   11   12
Base      G    T    C    A    G    C    T    A    G
Color   black red  blue green black blue red green black

Read:     A    T    C    G    T    C    A    G    C   (complement of template)
```

---

## 2. The .ab1 File Format

ABI/Applied Biosystems instruments save chromatogram data as `.ab1` files (also called **trace files**).

An `.ab1` file contains:

| Component | Description |
|-----------|-------------|
| **Raw traces** | Four channels of fluorescence intensity (one per base) |
| **Processed traces** | Signal after baseline subtraction, color deconvolution, normalization |
| **Base calls** | The sequence of called bases |
| **Peak locations** | Scan position where each base was called |
| **Quality values** | Phred quality score per base |
| **Metadata** | Instrument model, run date, sample name, primer, etc. |

BioPython reads `.ab1` files via `Bio.SeqIO` with format `"abi"`.

In [ ]:
# Setup
from Bio import SeqIO
from Bio.Seq import Seq
import numpy as np
import matplotlib.pyplot as plt

# If you have real .ab1 files, set this path accordingly:
# DATA_DIR = "path/to/your/chromatograms/"

print("All imports successful.")

### 2.1 Reading an .ab1 File

In [ ]:
def read_ab1(filepath):
    """
    Read an .ab1 chromatogram file and print a summary.
    Returns the SeqRecord.
    """
    record = SeqIO.read(filepath, "abi")

    print(f"File:            {filepath}")
    print(f"Sample ID:       {record.id}")
    print(f"Sequence length: {len(record.seq)} bp")
    print(f"First 60 bases:  {record.seq[:60]}")

    # Quality scores
    quals = record.letter_annotations["phred_quality"]
    print(f"Mean Phred:      {np.mean(quals):.1f}")
    print(f"Median Phred:    {np.median(quals):.1f}")

    return record

# ----- Uncomment when you have a real file: -----
# record = read_ab1(DATA_DIR + "sample_forward.ab1")

# For this tutorial we will also build synthetic examples
# so the notebook runs without data files.

### 2.2 What is Inside the Record?

The `SeqRecord` returned by BioPython stores traces in `record.annotations["abif_raw"]`.
Key tags:

| Tag | Content |
|-----|---------|
| `DATA9`-`DATA12` | Processed trace channels (order depends on dye set) |
| `DATA1`-`DATA4` | Raw trace channels |
| `PLOC1` / `PLOC2` | Peak locations (scan numbers) |
| `PBAS1` / `PBAS2` | Called bases (raw / edited) |
| `PCON1` / `PCON2` | Quality values |
| `FWO_1` | Base order for the four channels (e.g. `b'GATC'`) |

In [ ]:
def inspect_ab1_channels(record):
    """Show metadata stored in an .ab1 record."""
    raw = record.annotations.get("abif_raw", {})

    # Base order tells us which channel corresponds to which nucleotide
    base_order = raw.get("FWO_1", b"GATC").decode()
    print(f"Channel base order: {base_order}")

    # Processed trace data
    channels = {}
    for i, base in enumerate(base_order):
        tag = f"DATA{9 + i}"  # DATA9, DATA10, DATA11, DATA12
        trace = raw.get(tag, ())
        channels[base] = trace
        print(f"  Channel {base} ({tag}): {len(trace)} data points")

    # Peak locations
    peak_locs = raw.get("PLOC1", ())
    print(f"  Peak locations: {len(peak_locs)} called bases")

    return channels, peak_locs

# Uncomment with real data:
# channels, peaks = inspect_ab1_channels(record)

---

## 3. Chromatogram Visualization

Visualizing the four fluorescence traces is the most informative way to assess sequencing
quality. Good chromatograms have:
- Well-separated, evenly-spaced peaks
- Low baseline noise
- Consistent peak heights

We will first build a synthetic chromatogram for demonstration, then show how to
plot a real `.ab1` file.

In [ ]:
def make_synthetic_trace(sequence, peak_spacing=15, peak_width=4.0, noise_level=0.05):
    """
    Generate synthetic chromatogram traces for demonstration.

    Returns:
        traces: dict with keys A, T, G, C -> numpy arrays
        peak_positions: list of int scan positions for each base
    """
    n_bases = len(sequence)
    n_points = (n_bases + 2) * peak_spacing
    x = np.arange(n_points)

    traces = {b: np.zeros(n_points) for b in "ATGC"}
    peak_positions = []

    rng = np.random.default_rng(42)

    for i, base in enumerate(sequence.upper()):
        center = (i + 1) * peak_spacing
        peak_positions.append(center)
        # Gaussian peak for the called base
        amplitude = 0.8 + 0.2 * rng.random()
        if base in traces:
            traces[base] += amplitude * np.exp(-0.5 * ((x - center) / peak_width) ** 2)

    # Add background noise
    for b in traces:
        traces[b] += noise_level * rng.random(n_points)
        traces[b] = np.clip(traces[b], 0, None)

    return traces, peak_positions


demo_seq = "ATGCTAGCGATCGATCGATCGATCGATCGA"
traces, peaks = make_synthetic_trace(demo_seq)

print(f"Sequence:    {demo_seq}")
print(f"Trace length: {len(traces['A'])} scan points")
print(f"Bases called: {len(peaks)}")

In [ ]:
# Standard colors used for chromatograms
TRACE_COLORS = {"A": "green", "C": "blue", "G": "black", "T": "red"}


def plot_chromatogram(traces, peak_positions, sequence, start=0, end=None, title="Chromatogram"):
    """
    Plot a four-color chromatogram trace.

    Parameters:
        traces: dict of base -> numpy array of fluorescence values
        peak_positions: list of scan positions per called base
        sequence: string of called bases
        start, end: base index range to display (default: full sequence)
    """
    if end is None:
        end = len(sequence)

    # Determine scan range
    scan_start = max(0, peak_positions[start] - 10)
    scan_end = min(len(traces["A"]), peak_positions[min(end, len(peak_positions)) - 1] + 10)

    fig, ax = plt.subplots(figsize=(14, 3.5))

    x = np.arange(scan_start, scan_end)
    for base, color in TRACE_COLORS.items():
        ax.plot(x, traces[base][scan_start:scan_end], color=color, linewidth=0.8, label=base)

    # Label called bases at peak positions
    for i in range(start, min(end, len(peak_positions))):
        pos = peak_positions[i]
        base = sequence[i]
        color = TRACE_COLORS.get(base, "gray")
        ax.text(pos, -0.07 * ax.get_ylim()[1] if ax.get_ylim()[1] else -0.05,
                base, ha="center", va="top", fontsize=7, fontweight="bold", color=color)

    ax.set_xlabel("Scan position")
    ax.set_ylabel("Fluorescence intensity")
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=8, ncol=4)
    ax.set_xlim(scan_start, scan_end)
    plt.tight_layout()
    plt.show()


plot_chromatogram(traces, peaks, demo_seq, start=0, end=30, title="Synthetic Chromatogram")

In [ ]:
def plot_ab1_chromatogram(record, start=0, end=None, title=None):
    """
    Plot chromatogram directly from a BioPython .ab1 SeqRecord.
    """
    raw = record.annotations["abif_raw"]
    base_order = raw.get("FWO_1", b"GATC").decode()

    # Extract processed traces
    channels = {}
    for i, base in enumerate(base_order):
        channels[base] = np.array(raw[f"DATA{9 + i}"])

    peak_locs = list(raw["PLOC1"])
    sequence = str(record.seq)

    if title is None:
        title = f"Chromatogram: {record.id}"

    plot_chromatogram(channels, peak_locs, sequence, start=start,
                      end=end if end else len(sequence), title=title)

# Uncomment with real data:
# plot_ab1_chromatogram(record, start=50, end=120)

---

## 4. Phred Quality Scores and Base Calling

### 4.1 Phred Scores

Phred quality scores quantify the probability that a base call is **wrong**:

$$Q = -10 \times \log_{10}(P_{\text{error}})$$

| Phred Score | Error Probability | Accuracy |
|:-----------:|:-----------------:|:--------:|
| 10 | 1 in 10 | 90% |
| 20 | 1 in 100 | 99% |
| 30 | 1 in 1,000 | 99.9% |
| 40 | 1 in 10,000 | 99.99% |
| 50 | 1 in 100,000 | 99.999% |

**Practical guidelines:**
- Q < 20: unreliable -- consider trimming or re-sequencing
- Q 20-30: acceptable for most applications
- Q >= 30: high confidence

### 4.2 How Base Calling Works

Base-calling software (e.g. KB Basecaller, `phred`) performs:
1. **Baseline correction** -- subtract background fluorescence
2. **Color deconvolution** -- separate overlapping dye spectra ("spectral calibration")
3. **Peak detection** -- locate local maxima
4. **Spacing model** -- predict expected peak positions to handle compressions/expansions
5. **Quality assignment** -- evaluate peak shape, resolution, signal-to-noise to assign Phred scores

In [ ]:
def make_synthetic_qualities(n_bases, good_start=20, good_end=None, rng_seed=42):
    """
    Simulate realistic Phred quality profile for a Sanger read.
    Qualities ramp up at the start, plateau in the middle, and decay at the end.
    """
    if good_end is None:
        good_end = n_bases - 40
    rng = np.random.default_rng(rng_seed)

    quals = np.zeros(n_bases)
    for i in range(n_bases):
        if i < good_start:
            # Ramp-up region (primer / dye blobs)
            base_q = 8 + (30 - 8) * (i / good_start)
        elif i < good_end:
            # High-quality plateau
            base_q = 38 + 7 * rng.random()
        else:
            # Decay region
            frac = (i - good_end) / (n_bases - good_end)
            base_q = 38 * (1 - frac) + 5 * frac
        quals[i] = max(0, base_q + rng.normal(0, 3))

    return np.round(quals).astype(int)


# Simulate a 750-base read
synth_quals = make_synthetic_qualities(750)
print(f"Simulated {len(synth_quals)} quality scores")
print(f"Mean Q: {np.mean(synth_quals):.1f}  |  Median Q: {np.median(synth_quals):.0f}")
print(f"Min Q: {np.min(synth_quals)}  |  Max Q: {np.max(synth_quals)}")

In [ ]:
def plot_quality_scores(qualities, title="Phred Quality Scores"):
    """
    Plot quality scores along the read with color-coded bars.
    Green >= Q30, orange Q20-29, red < Q20.
    """
    positions = np.arange(1, len(qualities) + 1)
    colors = ["red" if q < 20 else "orange" if q < 30 else "green" for q in qualities]

    fig, ax = plt.subplots(figsize=(13, 3.5))
    ax.bar(positions, qualities, color=colors, width=1.0, edgecolor="none")
    ax.axhline(y=20, color="orange", linestyle="--", alpha=0.7, label="Q20 threshold")
    ax.axhline(y=30, color="green", linestyle="--", alpha=0.7, label="Q30 threshold")

    ax.set_xlabel("Base position")
    ax.set_ylabel("Phred quality")
    ax.set_title(title)
    ax.set_xlim(0, len(qualities) + 1)
    ax.set_ylim(0, max(qualities) + 5)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    # Summary
    low = sum(1 for q in qualities if q < 20)
    mid = sum(1 for q in qualities if 20 <= q < 30)
    high = sum(1 for q in qualities if q >= 30)
    total = len(qualities)
    print(f"Quality distribution:")
    print(f"  Low  (Q < 20):  {low:4d} bases ({100 * low / total:.1f}%)")
    print(f"  Med  (Q 20-29): {mid:4d} bases ({100 * mid / total:.1f}%)")
    print(f"  High (Q >= 30): {high:4d} bases ({100 * high / total:.1f}%)")


plot_quality_scores(synth_quals, "Synthetic Read -- Quality Profile")

---

## 5. Quality Trimming

Sanger reads typically have:
- **Low quality at the 5' end** -- primer sequence, dye blobs, signal instability
- **Low quality at the 3' end** -- signal decay after ~700-900 bp

Before downstream analysis, we trim these unreliable flanks.

### Sliding-Window Trimming

Move a window of fixed size across the quality array.  
The usable region starts at the first window whose mean quality >= threshold and ends at the last such window.

In [ ]:
def trim_by_quality(qualities, min_quality=20, window=10):
    """
    Find the longest internal region where sliding-window
    average quality >= min_quality.

    Returns (start, end) as 0-based half-open interval.
    """
    n = len(qualities)

    # Find 5' trim point
    start = 0
    for i in range(n - window):
        if np.mean(qualities[i:i + window]) >= min_quality:
            start = i
            break

    # Find 3' trim point
    end = n
    for i in range(n - 1, window, -1):
        if np.mean(qualities[i - window:i]) >= min_quality:
            end = i
            break

    return start, end


start, end = trim_by_quality(synth_quals, min_quality=20, window=10)
print(f"Original length: {len(synth_quals)} bp")
print(f"Trim region:     {start} - {end}  ({end - start} bp kept)")
print(f"Removed:         {len(synth_quals) - (end - start)} bp")

In [ ]:
def plot_trimming(qualities, start, end, title="Trimmed Read"):
    """Visualize trim boundaries on the quality profile."""
    positions = np.arange(1, len(qualities) + 1)

    fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)

    # Top: original with trim lines
    axes[0].bar(positions, qualities, width=1, color="steelblue", alpha=0.7)
    axes[0].axvline(x=start + 1, color="red", linestyle="--", label=f"Trim start ({start + 1})")
    axes[0].axvline(x=end, color="red", linestyle="--", label=f"Trim end ({end})")
    axes[0].axhline(y=20, color="orange", linestyle=":", alpha=0.5)
    axes[0].set_ylabel("Quality")
    axes[0].set_title("Original read with trim boundaries")
    axes[0].legend(fontsize=8)

    # Bottom: trimmed
    trimmed_q = qualities[start:end]
    trimmed_pos = np.arange(start + 1, end + 1)
    colors = ["red" if q < 20 else "orange" if q < 30 else "green" for q in trimmed_q]
    axes[1].bar(trimmed_pos, trimmed_q, width=1, color=colors, edgecolor="none")
    axes[1].axhline(y=20, color="orange", linestyle=":", alpha=0.5)
    axes[1].set_xlabel("Position (bp)")
    axes[1].set_ylabel("Quality")
    axes[1].set_title("After trimming")

    plt.tight_layout()
    plt.show()


plot_trimming(synth_quals, start, end)

In [ ]:
def trim_ab1_record(record, min_quality=20, window=10):
    """
    Trim an .ab1 SeqRecord by quality and return the trimmed SeqRecord.
    """
    quals = record.letter_annotations["phred_quality"]
    start, end = trim_by_quality(quals, min_quality, window)
    trimmed = record[start:end]

    print(f"Trimmed {record.id}: {len(record)} -> {len(trimmed)} bp  "
          f"(removed {start} from 5' and {len(record) - end} from 3')")
    return trimmed, start, end

# Uncomment with real data:
# trimmed_rec, s, e = trim_ab1_record(record)

---

## 6. Identifying Mixed Bases and Heterozygous Positions

### Why Do Double Peaks Occur?

At some positions the chromatogram shows **two overlapping peaks** of comparable height.
Common causes:

| Cause | Pattern |
|-------|---------|
| **Heterozygosity** | Double peaks at specific positions; rest of trace is clean |
| **Mixed template** | Double peaks throughout (two different templates) |
| **PCR artifacts** | Double peaks after a homopolymer or repeat region |
| **Contamination** | Degraded trace quality; multiple overlapping signals |

### IUPAC Ambiguity Codes

When the base caller cannot decide between two nucleotides, it may use IUPAC ambiguity codes:

| Code | Bases | Mnemonic |
|:----:|:-----:|:--------:|
| R | A or G | pu**R**ine |
| Y | C or T | p**Y**rimidine |
| S | G or C | **S**trong (3 H-bonds) |
| W | A or T | **W**eak (2 H-bonds) |
| K | G or T | **K**eto |
| M | A or C | a**M**ino |
| N | any | u**N**known |

In [ ]:
def make_heterozygous_trace(sequence, het_positions, het_alt_bases,
                            peak_spacing=15, peak_width=4.0, noise=0.03):
    """
    Create a synthetic chromatogram with heterozygous (double-peak) positions.

    het_positions: list of 0-based positions that are heterozygous
    het_alt_bases: dict mapping position -> alternative base
    """
    n_bases = len(sequence)
    n_points = (n_bases + 2) * peak_spacing
    x = np.arange(n_points)
    traces = {b: np.zeros(n_points) for b in "ATGC"}
    peak_positions = []
    rng = np.random.default_rng(7)

    for i, base in enumerate(sequence.upper()):
        center = (i + 1) * peak_spacing
        peak_positions.append(center)

        if i in het_positions:
            # Two peaks of roughly equal height
            alt = het_alt_bases[i]
            amp1 = 0.5 + 0.1 * rng.random()
            amp2 = 0.45 + 0.1 * rng.random()
            traces[base] += amp1 * np.exp(-0.5 * ((x - center) / peak_width) ** 2)
            traces[alt] += amp2 * np.exp(-0.5 * ((x - center) / peak_width) ** 2)
        else:
            amp = 0.8 + 0.2 * rng.random()
            traces[base] += amp * np.exp(-0.5 * ((x - center) / peak_width) ** 2)

    for b in traces:
        traces[b] += noise * rng.random(n_points)

    return traces, peak_positions


het_seq = "ATGCTAGCAATCGAACGATCG"
het_pos = {5: "A", 12: "T", 17: "C"}  # positions with two bases
het_traces, het_peaks = make_heterozygous_trace(
    het_seq, list(het_pos.keys()), het_pos
)

plot_chromatogram(het_traces, het_peaks, het_seq,
                  title="Chromatogram with Heterozygous Positions (5, 12, 17)")
print("Look for double peaks at positions 6, 13, and 18 (1-based).")

In [ ]:
IUPAC_AMBIGUITY = {
    frozenset("AG"): "R", frozenset("CT"): "Y",
    frozenset("GC"): "S", frozenset("AT"): "W",
    frozenset("GT"): "K", frozenset("AC"): "M",
}


def detect_heterozygous(traces, peak_positions, sequence, ratio_threshold=0.35):
    """
    Detect positions where a secondary peak is significant (potential heterozygosity).

    ratio_threshold: minimum secondary/primary peak ratio to flag.
    """
    results = []

    for i, (pos, base) in enumerate(zip(peak_positions, sequence.upper())):
        # Get fluorescence at peak position for each channel
        intensities = {}
        for b in "ATGC":
            idx = min(pos, len(traces[b]) - 1)
            intensities[b] = traces[b][idx]

        primary = intensities.get(base, 0)
        if primary == 0:
            continue

        # Check secondary peaks
        for alt_base, alt_val in intensities.items():
            if alt_base == base:
                continue
            ratio = alt_val / primary
            if ratio >= ratio_threshold:
                pair = frozenset([base, alt_base])
                iupac = IUPAC_AMBIGUITY.get(pair, "N")
                results.append({
                    "position": i + 1,
                    "called_base": base,
                    "alt_base": alt_base,
                    "ratio": ratio,
                    "iupac": iupac,
                })

    return results


het_hits = detect_heterozygous(het_traces, het_peaks, het_seq, ratio_threshold=0.35)

print(f"{'Pos':>4}  {'Called':>6}  {'Alt':>4}  {'Ratio':>6}  {'IUPAC':>6}")
print("-" * 35)
for h in het_hits:
    print(f"{h['position']:4d}  {h['called_base']:>6}  {h['alt_base']:>4}  "
          f"{h['ratio']:6.2f}  {h['iupac']:>6}")

In [ ]:
def find_low_quality_regions(qualities, threshold=20, min_run=5):
    """
    Find contiguous stretches of low quality.

    Returns list of (start, end) tuples (0-based, half-open).
    """
    regions = []
    in_low = False
    run_start = 0

    for i, q in enumerate(qualities):
        if q < threshold:
            if not in_low:
                run_start = i
                in_low = True
        else:
            if in_low:
                if i - run_start >= min_run:
                    regions.append((run_start, i))
                in_low = False

    # Handle run extending to end
    if in_low and len(qualities) - run_start >= min_run:
        regions.append((run_start, len(qualities)))

    return regions


low_regions = find_low_quality_regions(synth_quals, threshold=20, min_run=5)
print(f"Low-quality regions (Q < 20, >= 5 consecutive bases):")
for s, e in low_regions:
    print(f"  Positions {s + 1}-{e} ({e - s} bp), mean Q = {np.mean(synth_quals[s:e]):.1f}")

---

## 7. Assembly of Multiple Sanger Reads

A single Sanger read covers ~700-1000 bp. For longer targets you need **multiple overlapping reads** using different primers. This is called **primer walking** or **shotgun assembly**.

### Concepts

```
Target region (e.g. 3 kb plasmid insert):
========================================

Forward (primer F1):  --------->
Forward (primer F2):       --------->
Forward (primer F3):             --------->
Reverse (primer R1):                   <---------
Reverse (primer R2):             <---------
Reverse (primer R3):       <---------

Assembly:            ============================
                     overlap    overlap    overlap
```

### Assembly Steps

1. **Quality trim** each read
2. **Reverse-complement** reverse reads
3. **Align** overlapping regions (pairwise or multiple alignment)
4. **Build consensus** -- at each position, pick the base with the highest quality; flag disagreements
5. **Inspect** remaining ambiguities manually

### Tools for Sanger Assembly

| Tool | Description |
|------|-------------|
| **Sequencher** | Commercial; gold standard for Sanger assembly |
| **CodonCode Aligner** | Commercial; focus on mutation detection |
| **Geneious** | Commercial; general bioinformatics workbench |
| **CAP3** | Free; classic overlap-layout-consensus assembler |
| **Staden Gap4/5** | Free; older but powerful |
| **UGENE** | Free; open-source GUI-based bioinformatics suite |

In [ ]:
def simple_overlap_consensus(seq1, qual1, seq2, qual2, min_overlap=20):
    """
    Build a consensus from two overlapping reads by finding the best
    overlap and choosing the higher-quality base at each position.

    This is a simplified demonstration -- real assemblers use
    dynamic programming alignment.
    """
    best_offset = -1
    best_matches = 0

    # Slide seq2 across seq1 to find the best overlap
    for offset in range(len(seq1) - min_overlap + 1):
        overlap_len = min(len(seq2), len(seq1) - offset)
        if overlap_len < min_overlap:
            continue
        matches = sum(1 for a, b in zip(seq1[offset:], seq2[:overlap_len]) if a == b)
        if matches > best_matches:
            best_matches = matches
            best_offset = offset

    if best_offset < 0:
        raise ValueError("No significant overlap found.")

    overlap_len = min(len(seq2), len(seq1) - best_offset)
    identity = best_matches / overlap_len
    print(f"Best overlap at offset {best_offset}, length {overlap_len}, "
          f"identity {identity:.1%}")

    # Build consensus
    consensus = list(seq1[:best_offset])  # non-overlapping part of seq1

    # Overlapping region: pick higher-quality base
    for i in range(overlap_len):
        idx1 = best_offset + i
        q1 = qual1[idx1] if idx1 < len(qual1) else 0
        q2 = qual2[i] if i < len(qual2) else 0
        consensus.append(seq1[idx1] if q1 >= q2 else seq2[i])

    # Non-overlapping tail of seq2
    if overlap_len < len(seq2):
        consensus.extend(seq2[overlap_len:])

    return "".join(consensus), best_offset, overlap_len


# Demo with two synthetic overlapping reads
read1 = "ATCGATCGATCGATCGATCGATCGATCG"
read2 =               "TCGATCGATCGATCGATCGTTAACCGG"
# overlap:             TCGATCGATCGATCGATCG

qual1 = [35] * len(read1)
qual2 = [38] * len(read2)

consensus, off, ovl = simple_overlap_consensus(read1, qual1, read2, qual2, min_overlap=10)
print(f"Read 1:    {read1}")
print(f"Read 2:    {' ' * off}{read2}")
print(f"Consensus: {consensus}")
print(f"Length:    {len(consensus)} bp")

---

## 8. Sanger vs NGS: When to Use Which?

Despite the throughput revolution of next-generation sequencing, Sanger sequencing
remains **indispensable** in many situations.

### Comparison

| Feature | Sanger | Illumina (NGS) | Long-read (PacBio/ONT) |
|---------|--------|----------------|------------------------|
| **Read length** | 700-1000 bp | 150-300 bp | 10-50+ kb |
| **Throughput** | 1-96 reads/run | Millions-billions | Thousands-millions |
| **Per-base accuracy** | 99.99% | ~99.9% | 85-99.9% |
| **Cost per base** | High | Very low | Medium |
| **Turnaround** | Same day | 1-3 days | Hours-days |
| **Library prep** | Simple (PCR + primer) | Complex | Medium |

### When Sanger is the Right Choice

1. **Confirmation/validation** -- verify a cloned construct, a CRISPR edit, or an NGS variant
2. **Small number of targets** -- genotyping 1-10 amplicons is faster and cheaper by Sanger
3. **Mixed/heterozygous samples** -- chromatogram gives direct visual evidence of mixed bases
4. **Clinical diagnostics** -- gold standard for variant confirmation in many CLIA labs
5. **Barcoding** -- species identification via COI, 16S, ITS, rbcL with a single read
6. **Low-throughput labs** -- no expensive sequencer required; send samples to a core facility

### When NGS is Better

1. **Whole-genome** or **whole-exome** sequencing
2. **Transcriptomics** (RNA-seq)
3. **Hundreds+ of targets** (amplicon panels)
4. **Metagenomics** (environmental DNA)
5. **Structural variant detection** (long reads)

---

## 9. Exporting Results

After quality assessment and trimming, export the clean sequence to FASTA for
downstream analysis (BLAST, alignment, phylogenetics).

In [ ]:
from Bio.SeqRecord import SeqRecord


def export_trimmed_fasta(record, output_path, min_quality=20, window=10):
    """
    Trim an .ab1 record by quality and write FASTA.
    """
    trimmed, start, end = trim_ab1_record(record, min_quality, window)
    trimmed.description = f"{record.id} trimmed bases {start + 1}-{end}"

    with open(output_path, "w") as fh:
        SeqIO.write(trimmed, fh, "fasta")

    print(f"Wrote {len(trimmed)} bp to {output_path}")
    return trimmed


# Uncomment with real data:
# export_trimmed_fasta(record, "trimmed_forward.fasta")

---

## 10. Complete Workflow Example

Bringing everything together: load, inspect, trim, detect problems, export.

In [ ]:
def full_chromatogram_workflow(ab1_path, min_quality=20, window=10):
    """
    Complete chromatogram analysis pipeline.
    """
    # 1. Read
    rec = SeqIO.read(ab1_path, "abi")
    quals = rec.letter_annotations["phred_quality"]
    print(f"=== {rec.id} ===")
    print(f"Raw length: {len(rec)} bp, mean Q: {np.mean(quals):.1f}")

    # 2. Quality plot
    plot_quality_scores(quals, title=f"{rec.id} -- Quality")

    # 3. Trim
    start, end = trim_by_quality(quals, min_quality, window)
    trimmed = rec[start:end]
    tq = trimmed.letter_annotations["phred_quality"]
    print(f"\nTrimmed: {start}-{end} ({end - start} bp), mean Q: {np.mean(tq):.1f}")

    # 4. Low-quality regions inside trimmed read
    low = find_low_quality_regions(tq, threshold=20, min_run=3)
    if low:
        print(f"\nWarning: {len(low)} low-quality region(s) inside trimmed read:")
        for s, e in low:
            print(f"  {s + start + 1}-{e + start} (Q_mean={np.mean(tq[s:e]):.0f})")
    else:
        print("\nNo internal low-quality regions detected.")

    # 5. Chromatogram view of first ~80 bases after trim start
    plot_ab1_chromatogram(rec, start=start, end=min(start + 80, end),
                          title=f"{rec.id} -- Bases {start + 1}-{min(start + 80, end)}")

    return trimmed


# Uncomment with real data:
# trimmed = full_chromatogram_workflow("path/to/sample.ab1")

---

## Exercises

### Exercise 1: Read an .ab1 File and Summarize (Difficulty: *)

Load an `.ab1` file (or use the synthetic data), print:
- sequence length
- first and last 20 bases
- mean, median, min, and max Phred quality
- number of bases with Q < 20

In [ ]:
# Exercise 1 -- your code here
# Use the synthetic quality array as stand-in:

# synth_seq = "A" * 750   # placeholder sequence
# synth_quals  (already defined above)



In [ ]:
# --- Solution ---
seq_len = len(synth_quals)
synth_seq = "".join(np.random.default_rng(0).choice(list("ATGC"), seq_len))

print(f"Sequence length: {seq_len} bp")
print(f"First 20 bases:  {synth_seq[:20]}")
print(f"Last 20 bases:   {synth_seq[-20:]}")
print(f"Mean Phred:      {np.mean(synth_quals):.1f}")
print(f"Median Phred:    {np.median(synth_quals):.0f}")
print(f"Min Phred:       {np.min(synth_quals)}")
print(f"Max Phred:       {np.max(synth_quals)}")
print(f"Bases with Q<20: {np.sum(np.array(synth_quals) < 20)}")

### Exercise 2: Plot a Chromatogram Window (Difficulty: *)

Using `make_synthetic_trace`, generate a chromatogram for the sequence
`"GATTACAGATCGCTAGCTAGCTAGCTAGCTAG"` and plot bases 5 to 25.

In [ ]:
# Exercise 2 -- your code here


In [ ]:
# --- Solution ---
ex2_seq = "GATTACAGATCGCTAGCTAGCTAGCTAGCTAG"
ex2_traces, ex2_peaks = make_synthetic_trace(ex2_seq)
plot_chromatogram(ex2_traces, ex2_peaks, ex2_seq, start=5, end=25,
                  title="Exercise 2: Bases 6-25")

### Exercise 3: Identify Low-Quality Regions (Difficulty: **)

Given the `synth_quals` array, find all contiguous regions of at least 8 bases
where every quality score is below 25. Report the start, end, length, and mean Q
for each region.

In [ ]:
# Exercise 3 -- your code here


In [ ]:
# --- Solution ---
regions_ex3 = find_low_quality_regions(synth_quals, threshold=25, min_run=8)
print(f"Found {len(regions_ex3)} low-quality region(s) (Q < 25, >= 8 bp):")
for s, e in regions_ex3:
    print(f"  Positions {s + 1}-{e}: {e - s} bp, mean Q = {np.mean(synth_quals[s:e]):.1f}")

### Exercise 4: Detect Heterozygous Positions (Difficulty: **)

Create a synthetic chromatogram for `"AATCGGCCTTAAGGCC"` with heterozygous
positions at indices 3 (alt=A), 8 (alt=G), and 13 (alt=T). Use `detect_heterozygous`
to find them and report the IUPAC code for each.

In [ ]:
# Exercise 4 -- your code here


In [ ]:
# --- Solution ---
ex4_seq = "AATCGGCCTTAAGGCC"
ex4_hets = {3: "A", 8: "G", 13: "T"}
ex4_traces, ex4_peaks = make_heterozygous_trace(
    ex4_seq, list(ex4_hets.keys()), ex4_hets
)

plot_chromatogram(ex4_traces, ex4_peaks, ex4_seq,
                  title="Exercise 4: Heterozygous positions 4, 9, 14 (1-based)")

hits = detect_heterozygous(ex4_traces, ex4_peaks, ex4_seq, ratio_threshold=0.3)
print(f"Detected {len(hits)} heterozygous position(s):")
for h in hits:
    print(f"  Pos {h['position']}: {h['called_base']}/{h['alt_base']} "
          f"(ratio {h['ratio']:.2f}) -> IUPAC: {h['iupac']}")

### Exercise 5: Trim and Compare Two Reads (Difficulty: ***)

Simulate two quality profiles: one "good" read (high plateau from base 15 to 680) and
one "bad" read (plateau only from base 30 to 400). Trim both at Q >= 20 and report:
- trimmed length for each
- percentage of bases retained
- which read would you trust more, and why?

In [ ]:
# Exercise 5 -- your code here


In [ ]:
# --- Solution ---
good_quals = make_synthetic_qualities(750, good_start=15, good_end=680, rng_seed=1)
bad_quals = make_synthetic_qualities(750, good_start=30, good_end=400, rng_seed=2)

gs, ge = trim_by_quality(good_quals, 20, 10)
bs, be = trim_by_quality(bad_quals, 20, 10)

good_len = ge - gs
bad_len = be - bs

print("Good read:")
print(f"  Trimmed: {gs}-{ge} ({good_len} bp, {100 * good_len / 750:.1f}% retained)")
print(f"  Mean Q (trimmed): {np.mean(good_quals[gs:ge]):.1f}")

print("\nBad read:")
print(f"  Trimmed: {bs}-{be} ({bad_len} bp, {100 * bad_len / 750:.1f}% retained)")
print(f"  Mean Q (trimmed): {np.mean(bad_quals[bs:be]):.1f}")

print("\nThe good read retains more high-quality bases and covers a longer region.")
print("The bad read's signal decayed early, yielding a shorter usable sequence.")
print("For reliable results, the good read is clearly preferable.")

# Visualize both
fig, axes = plt.subplots(1, 2, figsize=(14, 3), sharey=True)
for ax, q, s, e, label in [(axes[0], good_quals, gs, ge, "Good read"),
                            (axes[1], bad_quals, bs, be, "Bad read")]:
    ax.bar(range(1, len(q) + 1), q, width=1, color="steelblue", alpha=0.6)
    ax.axvline(s + 1, color="red", linestyle="--")
    ax.axvline(e, color="red", linestyle="--")
    ax.axhline(20, color="orange", linestyle=":")
    ax.set_title(label)
    ax.set_xlabel("Position")
axes[0].set_ylabel("Phred Q")
plt.tight_layout()
plt.show()

---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| Sanger method | Dideoxy chain termination + capillary electrophoresis |
| .ab1 format | Contains traces, base calls, quality scores, metadata |
| BioPython | `SeqIO.read(path, "abi")` gives a full SeqRecord |
| Quality scores | Phred Q = -10 log10(P_error); Q >= 30 is high quality |
| Trimming | Sliding window removes low-quality 5'/3' ends |
| Heterozygosity | Double peaks in chromatogram; use IUPAC codes |
| Assembly | Overlap multiple reads; use highest-quality base at each position |
| Sanger vs NGS | Sanger for validation, few targets; NGS for high-throughput |

---

## Resources

- [BioPython ABI parsing documentation](https://biopython.org/wiki/ABI_chromatogram)
- [Phred quality scores (Wikipedia)](https://en.wikipedia.org/wiki/Phred_quality_score)
- [IUPAC nucleotide codes](https://www.bioinformatics.org/sms/iupac.html)
- [FinchTV](https://digitalworldbiology.com/FinchTV) -- free chromatogram viewer
- [4Peaks](https://nucleobytes.com/4peaks/) -- Mac chromatogram viewer
- Sanger F, Nicklen S, Coulson AR (1977). DNA sequencing with chain-terminating inhibitors. *PNAS* 74:5463-5467

---

[← Previous: Nucleic Acid Structure](../../Tier_2_Core_Bioinformatics/08_Nucleic_Acid_Structure/01_nucleic_acid_structure.ipynb) | [Next: Sequence Motifs and Protein Domains →](../../Tier_2_Core_Bioinformatics/10_Sequence_Motifs_and_Domains/01_sequence_motifs_and_domains.ipynb)